# Stage 4: Find promising layers and test the predictions

We have checked the baseline and the intervention mechanism. This stage asks:
**Which layers support forget answers more than retain answers, and do their scores predict actual effects?**

1. Calculate a score for every layer on **128 forget + 128 retain localization questions**.
2. Compare rankings from two balanced halves of those questions.
3. Fix the top-forget, top-selective, bottom-forget, and five random layer pairs.
4. Run **Control A** on **16 forget + 16 retain development questions**:
   weaken each layer separately at strengths **0.05 and 0.5**, then compare predicted and measured effects.

All weights stay frozen. Final-test questions are reserved for later.
Poor ranking stability or poor predictions are findings to discuss, not a reason to change the formula.

**To run:** select a **T4 GPU**, choose **Run all**, upload **unlearning-stage4.zip**, and allow `HF_TOKEN` access.
The ZIP includes the verified data and Stage 2/3 evidence. No earlier stage needs rerunning.
Download **stage4-results.zip** at the end, including if interrupted or incomplete.

The measured Stage 3 timings suggest roughly **5-10 minutes of model work**, plus setup and model download.
Actual time depends on Colab and prompt lengths. Progress is saved after every measurement.


## 1. Upload the Stage 4 bundle

Select `dist/unlearning-stage4.zip` from the project folder on your computer.


In [ ]:
import hashlib
import io
import json
import os
from pathlib import Path, PurePosixPath
import subprocess
import sys
import zipfile
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1 or 'unlearning-stage4.zip' not in uploaded:
    raise ValueError('Select unlearning-stage4.zip from the project dist folder.')
project = Path('/content/unlearning_stage4')
project.mkdir(exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded['unlearning-stage4.zip'])) as archive:
    names = archive.namelist()
    if len(names) != len(set(names)):
        raise ValueError('Duplicate archive entries.')
    bundle = json.loads(archive.read('bundle_manifest.json'))
    if set(names) != set(bundle['files']) | {'bundle_manifest.json'}:
        raise ValueError('Archive contents do not match the bundle manifest.')
    for name in names:
        relative = PurePosixPath(name)
        if relative.is_absolute() or '..' in relative.parts or '\\' in name:
            raise ValueError('Unexpected archive path.')
        target = (project / name).resolve()
        if project.resolve() not in target.parents:
            raise ValueError('Archive path is outside the project.')
        content = archive.read(name)
        if name != 'bundle_manifest.json' and hashlib.sha256(content).hexdigest() != bundle['files'][name]:
            raise ValueError('Archive checksum mismatch: ' + name)
        if name.startswith('inputs/') and target.exists() and target.read_bytes() != content:
            raise ValueError('Existing inputs differ. Use a fresh Colab runtime.')
    for name in names:
        target = project / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(name))
del uploaded
os.chdir(project)
print('Stage 4 code and verified inputs ready.')


## 2. Install and check the code and inputs

The tests use synthetic data and tiny random models. They check calculation, data separation,
and resuming after an interruption. These test results are not research findings.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--only-binary=:all:',
                '-r', 'requirements-stage4-colab.txt'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'], check=True)
subprocess.run([sys.executable, '-m', 'unlearning', 'doctor',
                '--output', 'outputs/stage4_environment.json'], check=True)
environment = json.loads(Path('outputs/stage4_environment.json').read_text())
if not environment['cuda_available']:
    raise RuntimeError('Select a T4 GPU runtime before continuing.')
subprocess.run([sys.executable, '-c',
    "from unlearning.baseline import read_settings; from unlearning.data import read_config; "
    "from unlearning.localization import read_localization_settings, stage4_inputs; "
    "stage4_inputs(read_config('configs/experiment.json'), read_settings('configs/baseline_prefill.json'), "
    "read_localization_settings('configs/localization.json'), 'inputs/data/prepared/full', "
    "'inputs/outputs/stage2/prefill', 'inputs/stage3'); print('Verified Stage 4 inputs.')"], check=True)


## 3. Optional: restore an interrupted Stage 4 run

On your first run, leave this cell unchanged. It does not ask for another upload.
To recover after losing a runtime, change `RESTORE_RESULTS` to `True` and upload your latest
`stage4-results.zip`. Existing results must be identical; use a fresh runtime if they conflict.
Records are checked before any GPU work resumes. Resume requires matching code, settings, and runtime versions.


In [ ]:
RESTORE_RESULTS = False
result_dir = Path('outputs/stage4/full')
if RESTORE_RESULTS:
    from unlearning.result_archives import restore_stage4_results
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError('Select exactly one saved Stage 4 results ZIP.')
    print(restore_stage4_results(next(iter(uploaded.values())), project))
    del uploaded
    # A prior failed measurement still needs review, but valid partial records can resume.
    from unlearning.localization_report import stage4_report
    restored = stage4_report(result_dir, make_plots=False)
    print('Verified saved records:', restored['completed_records'], '/', restored['expected_records'])
    if restored['status'] == 'review_required':
        print('The previous attempt needs attention. Read its saved failure before resuming.')


## 4. Run localization and Control A

`MAX_NEW = None` runs all remaining work: **256 localization gradients, 32 control gradients,
and 1,024 single-layer interventions**. It saves 1,312 records in total.
Set `MAX_NEW = 256` for shorter segments, then download a backup and rerun this cell to continue.

The layer pairs are fixed after all localization questions are complete, before the control outcomes.
A positive score predicts that weakening a layer lowers the correct-answer score.
Selectivity is the forget mean minus the retain mean. Negative scores are retained.


In [ ]:
import getpass
from google.colab import userdata
MAX_NEW = None

saved_summary = json.loads((result_dir / 'summary.json').read_text()) if (result_dir / 'summary.json').exists() else {}
if saved_summary.get('completed_records') == saved_summary.get('expected_records') and saved_summary.get('expected_records'):
    # Regenerate tables/figures only; completed model measurements are not repeated.
    completed = subprocess.run([sys.executable, '-m', 'unlearning', 'stage4-report', '--output', str(result_dir)])
else:
    try:
        os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()
    except Exception:
        os.environ['HF_TOKEN'] = getpass.getpass('Hugging Face read token (hidden): ').strip()
    if not os.environ['HF_TOKEN']:
        raise RuntimeError('No token supplied.')
    args = [sys.executable, '-m', 'unlearning', 'localize', '--output', str(result_dir)]
    if MAX_NEW is not None:
        args.extend(['--max-new', str(MAX_NEW)])
    completed = subprocess.run(args)
if completed.returncode:
    print('The command needs attention. Read the message above and download the saved evidence below.')
else:
    print('Segment finished. Review the summary and download a backup below.')


## 5. Read the results

**complete** means all measurements were saved and the report was produced. It does not mean the hypothesis succeeded.
**partial** means more work remains; rerun the previous cell with the same settings.
**review_required** means a technical failure or report issue needs attention.

The figures show layer means, agreement between two data halves, and predicted versus measured score drops.
In Control A, points near the diagonal indicate accurate predictions. Compare the small and larger strength.
These are changes in the correct-answer score, not changes in accuracy percentage points.


In [ ]:
from IPython.display import display, Image
summary_path = result_dir / 'summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text())
    print('Stage 4:', summary['status'])
    print('Saved:', summary['completed_records'], '/', summary['expected_records'])
    for selection in summary.get('selections', []):
        print(selection['name'], ': layers', selection['layers'])
    for name, stability in summary.get('stability', {}).items():
        print(name, 'half-to-half Spearman:', stability['spearman'], '| top-pair overlap:', stability['top_k_overlap'])
    if summary.get('last_attempt', {}).get('failure'):
        print(json.dumps(summary['last_attempt']['failure'], indent=2))
    print(summary['next_action'])
    for path in sorted((result_dir / 'figures').glob('*.png')):
        display(Image(filename=str(path), width=900))
else:
    print('No run report was written. Keep the error above and download the setup evidence below.')


## 6. Download a backup

Share `stage4-results.zip` for review. Download it after each segment and before disconnecting.
The archive contains the per-question records, frozen selections (when ready), figures, source, settings, and prior-stage evidence.
Stage 5 does not start automatically.


In [ ]:
archive_path = Path('/content/stage4-results.zip')
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(result_dir.rglob('*')):
        if path.is_file() and not path.name.endswith('.tmp'):
            archive.write(path, path.as_posix())
    for name in ('outputs/stage4_environment.json', 'bundle_manifest.json', 'STAGE4.md',
                 'requirements-colab.txt', 'requirements-stage4-colab.txt'):
        if Path(name).exists():
            archive.write(name)
    for directory, pattern in (('configs', '*.json'), ('src/unlearning', '*.py'), ('inputs/stage3', '*.json')):
        for path in sorted(Path(directory).glob(pattern)):
            archive.write(path, path.as_posix())
files.download(str(archive_path))
